# Staged PINN Solar — production run on Colab A100 (NOTEBOOK v3)

**How to use:** Runtime → Change runtime type → **A100 GPU, High-RAM**. Then **Runtime → Run all**. That is all.

* Runs the steps in order (default `e1`, `e2`, `e5`) with r04 / protocol v03, full batch, float64.
* Every finished unit is written **directly to Google Drive** (`MyDrive/StagedPINN/Results/<step>/<unit>.json`).
* **Interrupted? Run all again.** Finished units are skipped; only the unit in progress is redone.
* Each step trains in a child process: a CUDA fault kills only that process, which is restarted automatically.
* Dataset credentials come from the Colab secret **`sgarizado`** (Kaggle token).


In [ ]:
#@title 1. GPU and Google Drive
import os, subprocess, pathlib, json, time, sys
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip() or "NO GPU: Runtime -> Change runtime type -> A100")
from google.colab import drive
drive.mount("/content/drive")
DRIVE_RESULTS = pathlib.Path("/content/drive/MyDrive/StagedPINN/Results")
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
print("results on Drive:", DRIVE_RESULTS)


In [ ]:
#@title 2. Code (latest master) and dependencies
ROOT = pathlib.Path("/content/pinn-solar-staged")
if ROOT.exists():
    subprocess.run(["git", "-C", str(ROOT), "fetch", "-q", "origin", "master"], check=True)
    subprocess.run(["git", "-C", str(ROOT), "reset", "-q", "--hard", "origin/master"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "https://github.com/smindiol/pinn-solar-staged.git", str(ROOT)], check=True)
print("HEAD:", subprocess.run(["git", "-C", str(ROOT), "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "h5py", "scipy", "optuna", "kagglehub"], check=True)
SRC = str(ROOT / "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)


In [ ]:
#@title 3. Kaggle secret `sgarizado` and the dPV dataset
try:
    from google.colab import userdata
    secret = (userdata.get("sgarizado") or "").strip()
except Exception as e:
    secret = ""
    print("secret 'sgarizado' not readable:", type(e).__name__)
if secret.startswith("{"):                       # kaggle.json content
    k = json.loads(secret)
    os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = k["username"], k["key"]
elif secret:                                     # API token (KGAT...)
    os.environ["KAGGLE_API_TOKEN"] = secret
print("kaggle credentials:", "from secret 'sgarizado'" if secret else "none (public dataset / Colab cache)")
from Staged_PINN_Solar import data
DATA_ROOT = str(data.ensure_dataset(target="/content/dpv"))
print("data root:", DATA_ROOT, len(list(pathlib.Path(DATA_ROOT).glob("*.h5"))), "files")


In [ ]:
#@title 4. Configuration
STEPS = "e1,e2,e5"  #@param {type:"string"}
TRAIN_CHUNK_DEVICES = 16  #@param {type:"integer"}
MAX_RESTARTS_PER_STEP = 20  #@param {type:"integer"}
from Staged_PINN_Solar import config as C, steps, units as U
cfg = C.default().with_(full_batch=True, train_chunk_devices=TRAIN_CHUNK_DEVICES, wall_budget_seconds=1e9)
STEP_LIST = [s.strip() for s in STEPS.split(",") if s.strip()]

def done_units(step):
    return [u for u in U.enumerate_units(step, cfg) if steps.is_complete(steps.unit_path(step, u, False, DRIVE_RESULTS))]

for s in STEP_LIST:
    print(f"{s}: {len(done_units(s))}/{len(U.enumerate_units(s, cfg))} units already on Drive")


In [ ]:
#@title 5. Run (resumable: just Run all again after any interruption)
env = dict(os.environ, STAGED_PINN_WORKER="1", PYTHONPATH=SRC, STAGED_PINN_RESULTS=str(DRIVE_RESULTS))
env.pop("PYTORCH_CUDA_ALLOC_CONF", None)
payload_of = lambda step: json.dumps({"step": step, "units": None, "smoke": False, "root": DATA_ROOT,
                                       "out": str(DRIVE_RESULTS), "config": cfg.to_dict()})
for step in STEP_LIST:
    total = len(U.enumerate_units(step, cfg))
    stalled = 0
    for attempt in range(1, MAX_RESTARTS_PER_STEP + 1):
        before = len(done_units(step))
        if before == total:
            break
        print(f"=== {step}: {before}/{total} done, attempt {attempt} ===", flush=True)
        t0 = time.time()
        proc = subprocess.Popen([sys.executable, "-u", "-c",
                                 "import sys; from Staged_PINN_Solar import steps; steps._worker_main(sys.argv[1])",
                                 payload_of(step)], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        tail = []
        for line in proc.stdout:
            line = line.rstrip()
            tail = (tail + [line])[-15:]
            if ": written " in line or "skipped" not in line and line.startswith("["):
                print(line, flush=True)
        code_ = proc.wait()
        after = len(done_units(step))
        print(f"--- {step}: exit {code_}, {after}/{total} done, {(time.time()-t0)/3600:.2f} h this attempt", flush=True)
        if code_ != 0:
            print("\n".join(tail[-8:]), flush=True)
        stalled = stalled + 1 if after == before else 0
        if stalled >= 3:
            print(f"{step}: three attempts in a row without finishing a unit; moving on (see the error above)")
            break
print("finished:", {s: f"{len(done_units(s))}/{len(U.enumerate_units(s, cfg))}" for s in STEP_LIST})


In [ ]:
#@title 6. Summary of what is on Drive
rows = []
for s in STEP_LIST:
    for p in sorted((DRIVE_RESULTS / s).glob("*.json")):
        try:
            d = json.loads(p.read_text())
        except Exception:
            continue
        if not d.get("complete"):
            continue
        err = [v for v in (d.get("per_device_error") or {}).values() if v is not None]
        rows.append((s, d["unit"], d.get("diverged"), round(d.get("wall_seconds", 0) / 3600, 2),
                     round(float(sum(err) / len(err)), 3) if err else None))
print(f"{'step':5} {'unit':28} {'diverged':9} {'hours':6} mean PCE error %")
for r in rows:
    print(f"{r[0]:5} {r[1]:28} {str(r[2]):9} {r[3]:<6} {r[4]}")
